# A Scanner for Simple Arithmetic Expressions

This notebook shows how a scanner can be implemented using nothing but regular expressions &mdash;
no scanner generator like `ocamllex` is needed.  We use the module `Re` from the file `../lib/regex.ml`,
which offers the functions of Python's module `re` on top of the library PCRE2.  We build a scanner that
tokenizes simple arithmetic expressions such as
```
3 + 4 * 10 + 007 + (-20) * 2
```
into a sequence of tokens like `NUMBER`, `PLUS`, `TIMES`, and so on.

The general recipe consists of three steps:
1. We decide on the names of all token types that our scanner is supposed to recognize.
2. For every token name, we specify a regular expression that describes the set of strings that are to be
   recognized as tokens of this type.
3. We combine all of these regular expressions into a single *master pattern* and use this pattern to scan
   the input string from left to right.

## Type Checking

The functions in this notebook carry *type annotations*.  In OCaml, these annotations are checked by the
compiler before a cell is executed, so there is no need for a separate type checker like *basedpyright*.
A cell that contains a type error is not executed at all.

## Imports

We load the helper functions shared by all notebooks and the module `Re`.  Instead of Python's
`namedtuple`, we use an OCaml *record type* for our tokens, and instead of a Python *generator* we return a
*lazy sequence* of type `token Seq.t`.  Both are part of the OCaml language and its standard library.

In [1]:
#use "../lib/nb.ml";;
load_lib "regex.ml"

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()
- : unit = ()
val lib_dir : string = "../lib"
val loaded_libs : (string, unit) Hashtbl.t = <abstr>
val load_lib : string -> unit = <fun>


- : unit = ()


## Specifying the Tokens

The *token specification* below is a list of pairs.  The first component of every pair is the name of a
token, written in capital letters by convention.  The second component is a regular expression that
describes the strings which are to be recognized as tokens of this type.  The regular expressions are
written as *quoted strings* `{|...|}`, which correspond to Python's raw strings.

Three of the names occurring below do not name genuine tokens.  Instead, they serve a bookkeeping purpose:
- `NEWLINE` matches a single newline character.  Whenever this token is found, we have to increment our
  line counter.  The newline character itself is then discarded.
- `SKIP` matches blanks and tabs.  These characters only separate tokens and are therefore discarded.
- `MISMATCH` matches any single character that has not already been matched by one of the preceding
  regular expressions.  If this token is found, an unexpected character has occurred in the input and the
  scanner has to report an error.

**Note:** Like Python's `re` module, PCRE2 tries the alternatives of a regular expression from left to right and picks
the *first* one that matches at the current position&mdash;unlike some scanner generators, which pick the
*longest* match irrespective of order.  Therefore, the order of the pairs below is significant: more
specific patterns have to be listed before more general ones.

In [2]:
let token_specification : (string * string) list = [
    ("NUMBER",   {|0|[1-9][0-9]*|});
    ("PLUS",     {|\+|});
    ("MINUS",    {|-|});
    ("TIMES",    {|\*|});
    ("DIVIDE",   {|/|});
    ("LPAREN",   {|\(|});
    ("RPAREN",   {|\)|});
    ("NEWLINE",  {|\n|});
    ("SKIP",     {|[ \t]+|});
    ("MISMATCH", {|.|});
]

val token_specification : (string * string) list =
  [("NUMBER", "0|[1-9][0-9]*"); ("PLUS", "\\+"); ("MINUS", "-");
   ("TIMES", "\\*"); ("DIVIDE", "/"); ("LPAREN", "\\("); ("RPAREN", "\\)");
   ("NEWLINE", "\\n"); ("SKIP", "[ \\t]+"); ("MISMATCH", ".")]


## Building the Master Pattern

We combine the individual regular expressions into a single *master pattern*.  Every regular expression is
wrapped inside a *named group*, which is written as
```
(?P<NAME>regexp)
```
These named groups are then joined using the alternative operator `|`.  Later, once a match has been found,
we can ask which of the named groups is responsible for the match.

In [3]:
let master_pattern : string =
  String.concat "|" (List.map (fun (name, regexp) -> Printf.sprintf "(?P<%s>%s)" name regexp) token_specification)

val master_pattern : string =
  "(?P<NUMBER>0|[1-9][0-9]*)|(?P<PLUS>\\+)|(?P<MINUS>-)|(?P<TIMES>\\*)|(?P<DIVIDE>/)|(?P<LPAREN>\\()|(?P<RPAREN>\\))|(?P<NEWLINE>\\n)|(?P<SKIP>[ \\t]+)|(?P<MISMATCH>.)"


Let's take a look at the master pattern that has been generated:

In [4]:
master_pattern

- : string =
"(?P<NUMBER>0|[1-9][0-9]*)|(?P<PLUS>\\+)|(?P<MINUS>-)|(?P<TIMES>\\*)|(?P<DIVIDE>/)|(?P<LPAREN>\\()|(?P<RPAREN>\\))|(?P<NEWLINE>\\n)|(?P<SKIP>[ \\t]+)|(?P<MISMATCH>.)"


## Representing Tokens

In Python, the class `Token` is defined using `collections.namedtuple`, and every attribute can hold a value
of any type.  In OCaml, we define the *variant type* `kind` of all token names and the *record type*
`token` with the four fields `kind`, `value`, `line`, and `column`.  (The Python attribute `type` is called
`kind` here, since `type` is a keyword of OCaml.)

The value of a `NUMBER` token is an integer, while the value of every other token is a string.  In Python,
the attribute `value` therefore has the type `int | str`.  In OCaml, this is made explicit by the variant type
`value` with the constructors `Int` and `Str`.

**Type safety:** Only the names of the token specification are values of the type `kind`.  If we misspell a
token name in a `match` expression, e.g. `NUMBR`, the compiler reports an error, while Python would silently
compare `kind` with the string `'NUMBR'`.  Moreover, the compiler warns if a `match` on a `kind` forgets one
of the cases.  The function `kind_of_name` converts the name of a group into a `kind`.

In [5]:
type kind = NUMBER | PLUS | MINUS | TIMES | DIVIDE | LPAREN | RPAREN | NEWLINE | SKIP | MISMATCH

type value = Int of int | Str of string

type token = { kind : kind; value : value; line : int; column : int }

let kind_of_name (name : string) : kind =
  match name with
  | "NUMBER" -> NUMBER | "PLUS" -> PLUS | "MINUS" -> MINUS | "TIMES" -> TIMES | "DIVIDE" -> DIVIDE
  | "LPAREN" -> LPAREN | "RPAREN" -> RPAREN | "NEWLINE" -> NEWLINE | "SKIP" -> SKIP | "MISMATCH" -> MISMATCH
  | _ -> invalid_arg ("kind_of_name: " ^ name)

type kind =
    NUMBER
  | PLUS
  | MINUS
  | TIMES
  | DIVIDE
  | LPAREN
  | RPAREN
  | NEWLINE
  | SKIP
  | MISMATCH


type value = Int of int | Str of string


type token = { kind : kind; value : value; line : int; column : int; }


val kind_of_name : string -> kind = <fun>


## The Scanner

The function `tokenize` returns a *lazy sequence* of tokens, one for every token found in its argument `code`.
A value of type `token Seq.t` is a function that computes the next token only when it is asked for it, just
like a Python *generator*.

- `ln` (short for *line number*) counts the lines of the input, starting at $1$.  `line_start` stores the position at which
  the current line begins; it is needed to compute the column of a token.  Since OCaml variables can not be
  changed, `ln` and `line_start` are parameters of the local function `next`, which is called with new
  values whenever a newline is found.
- `Re.finditer` scans the string `code` for non-overlapping matches of `master_pattern`.  It returns a
  lazy sequence of *match objects*, one for every token that is found, in the order in which these tokens occur
  in the input. For a *match object* `mo`, we have the following functions:
  1. `Re.lastgroup mo` returns the name of the named group responsible
     for the match, i.e. the name of the token that has been found.  The result has the type `string option`,
     since in general there might be no named group that has matched.  Here, every alternative is a named
     group, so `Option.get` can be used.
  2. `Re.group mo 0` extracts the string that has been matched.
  3. `Re.start mo` returns the position at which the match begins.
  4. `Re.end_ mo` returns the position at which the match ends.
- `kind` is the kind of the token,
- `value` is the string that has been matched.
- `col` (short for *column*) is computed by subtracting `line_start` from the position at which the match begins.
- The token `NUMBER` is handled by converting the matched string into an `int`.
- The token `NEWLINE` updates `line_start`, increments the line counter `ln`, and then continues with the
  next match&mdash;no token is generated for a newline.
- The token `SKIP` is simply discarded by continuing with the next match.
- The token `MISMATCH` raises the exception `Failure` that reports the illegal character together with the line and
  column at which it was found.
- Every other match produces a `token` record, which is the next element of the sequence.

**Function `tokenize code`**
- *Input:* `code` is a string that contains arithmetic expressions.
- *Output:* A lazy sequence that produces one `token` for every token in `code`.  The exception `Failure` is raised if `code` contains an illegal character.

In [6]:
let tokenize (code : string) : token Seq.t =
  let rec next (matches : Re.match_ Seq.t) (ln : int) (line_start : int) : token Seq.t = fun () ->
    match matches () with
    | Seq.Nil -> Seq.Nil
    | Seq.Cons (mo, rest) ->
        let kind  = kind_of_name (Option.get (Re.lastgroup mo)) in
        let value = Re.group mo 0 in
        let col   = Re.start mo - line_start in   (* column of the token *)
        match kind with
        | NUMBER   -> Seq.Cons ({ kind; value = Int (int_of_string value); line = ln; column = col },
                                next rest ln line_start)
        | NEWLINE  -> next rest (ln + 1) (Re.end_ mo) ()
        | SKIP     -> next rest ln line_start ()
        | MISMATCH -> failwith (Printf.sprintf "%s: line %d, col %d." (repr value) ln col)
        | PLUS | MINUS | TIMES | DIVIDE | LPAREN | RPAREN ->
            Seq.Cons ({ kind; value = Str value; line = ln; column = col }, next rest ln line_start)
  in
  next (Re.finditer master_pattern code) 1 0

val tokenize : string -> token Seq.t = <fun>


## Trying it Out

We define a sample arithmetic expression that we want to tokenize.  Python's `print` shows a token as
`Token(type='NUMBER', value=3, line=1, column=0)`.  In OCaml, there is no generic function that turns an
arbitrary value into a string, so we write the function `show_token`, which produces the same format.

In [7]:
let data = "3 + 4 * 10 +\n 007 + (-20) * 2"

let show_kind (k : kind) : string =
  match k with
  | NUMBER -> "NUMBER" | PLUS -> "PLUS" | MINUS -> "MINUS" | TIMES -> "TIMES" | DIVIDE -> "DIVIDE"
  | LPAREN -> "LPAREN" | RPAREN -> "RPAREN" | NEWLINE -> "NEWLINE" | SKIP -> "SKIP" | MISMATCH -> "MISMATCH"

let show_token (t : token) : string =
  let value = match t.value with Int n -> string_of_int n | Str s -> repr s in
  Printf.sprintf "Token(kind=%s, value=%s, line=%d, column=%d)" (show_kind t.kind) value t.line t.column

val data : string = "3 + 4 * 10 +\n 007 + (-20) * 2"


val show_kind : kind -> string = <fun>


val show_token : token -> string = <fun>


We call `tokenize data` and print every token that is found.  Since `tokenize` returns a lazy sequence,
calling it does not scan anything yet; the scanning only happens while `Seq.iter` asks for the tokens, one
token at a time.

In [8]:
Seq.iter (fun tok -> print_endline (show_token tok)) (tokenize data)

Token(kind=NUMBER, value=3, line=1, column=0)
Token(kind=PLUS, value="+", line=1, column=2)
Token(kind=NUMBER, value=4, line=1, column=4)
Token(kind=TIMES, value="*", line=1, column=6)
Token(kind=NUMBER, value=10, line=1, column=8)
Token(kind=PLUS, value="+", line=1, column=11)
Token(kind=NUMBER, value=0, line=2, column=1)
Token(kind=NUMBER, value=0, line=2, column=2)
Token(kind=NUMBER, value=7, line=2, column=3)
Token(kind=PLUS, value="+", line=2, column=5)
Token(kind=LPAREN, value="(", line=2, column=7)
Token(kind=MINUS, value="-", line=2, column=8)
Token(kind=NUMBER, value=20, line=2, column=9)
Token(kind=RPAREN, value=")", line=2, column=11)
Token(kind=TIMES, value="*", line=2, column=13)
Token(kind=NUMBER, value=2, line=2, column=15)


- : unit = ()


The scanner also keeps track of the line and the column of every token when the input consists of several
lines.  Note that the column is reset to $0$ at the start of every line.

In [ ]:
Seq.iter (fun tok -> print_endline (show_token tok)) (tokenize "3 + 4 * 10 + 007 +\n(-20) * 2")

## Token Attributes

The scanner returns tokens as records of type `token`, which possess four distinct fields:
- The **`kind`** field indicates the type of the token.  It holds a value of the type `kind`, which
  corresponds to one of the names occurring in the token specification.
- The **`value`** field usually contains the recognized string.  However, as shown in the case of the
  token `NUMBER`, this value can be transformed before the token is constructed.
- The **`line`** field specifies the line number at which the token was found.
- The **`column`** field specifies the column at which the token starts.  Note that, unlike a counter
  that simply counts characters from the very beginning of the input, this column is reset to $0$ at the
  start of every line, which matches the way most editors report cursor positions.

## What Happens with an Unexpected Character?

Finally, let's see what happens if the input contains a character that is not covered by any of the
regular expressions in `token_specification`, for example the character `#`.

In [9]:
try
  Seq.iter (fun tok -> print_endline (show_token tok)) (tokenize "3 + 4 # 2")
with Failure error ->
  Printf.printf "Illegal character %s\n%!" error

Token(kind=NUMBER, value=3, line=1, column=0)
Token(kind=PLUS, value="+", line=1, column=2)
Token(kind=NUMBER, value=4, line=1, column=4)
Illegal character "#": line 1, col 6.


- : unit = ()
